In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

def prepare_fusion_dataset(bert_path, ped_path, static_path, output_path):
    print("1. Đang đọc dữ liệu...")
    df_bert = pd.read_csv(bert_path)
    df_ped = pd.read_csv(ped_path)
    df_static = pd.read_csv(static_path)
    
    # 2. Hợp nhất dữ liệu dựa trên cột ID
    df_merged = df_bert.merge(df_ped, on='ID').merge(df_static, on='ID')
    print(f"Tổng số bài tập sau khi gộp: {len(df_merged)}")
    
    # 3. Phân loại các cột đặc trưng
    # (Bạn hãy điều chỉnh tên cột cho khớp với file CSV thực tế của bạn)
    
    # Nhóm 1: Các đặc trưng đếm (Dùng Standard Scaler)
    # Ví dụ: word_count, sentence_length, Inference_Steps
    count_features = ['word_count', 'sentence_length'] 
    if 'Inference_Steps' in df_merged.columns: count_features.append('Inference_Steps')
        
    # Nhóm 2: Các đặc trưng sư phạm/lỗi sai (Dùng Min-Max Scaler)
    ped_features = ['LLM_Arithmetic', 'LLM_Procedural', 'LLM_Conceptual', 
                    'LLM_Lack_of_Sense', 'LLM_Misconception_Score']
    
    print("2. Đang tiến hành Chuẩn hóa Lai (Hybrid Scaling)...")
    # Standard Scaling
    std_scaler = StandardScaler()
    df_merged[count_features] = std_scaler.fit_transform(df_merged[count_features])
    
    # Min-Max Scaling
    mm_scaler = MinMaxScaler()
    df_merged[ped_features] = mm_scaler.fit_transform(df_merged[ped_features])
    
    # (Nhóm Vecto Q và BERT Embeddings được giữ nguyên)
    
    # 4. Lưu dữ liệu
    df_merged.to_csv(output_path, index=False)
    print(f"✅ Đã lưu tập dữ liệu chuẩn bị cho Fusion tại: {output_path}")
    
    return df_merged, count_features, ped_features

# Chạy thử
df_final, count_cols, ped_cols = prepare_fusion_dataset(
    "D:\IntelligentTesting\IntelligentTesting\Embedding\bert_embeddings_768d.csv", 
    "llm_misconceptions.csv", 
    "D:\IntelligentTesting\IntelligentTesting\ExtractFeature\full_features.csv", 
    "V_item_ready.csv"
)